# 02 — Organize Corpus & Validate Oracle

**Purpose.** Score every (sampled) real checkpoint with the DynaHug oracle container, build seed-selection views, and produce the disjoint train/eval split used to keep the FP study independent of calibration.

**Inputs / outputs**
- `real_benign_corpus/all/` (>=100 real models)
- `regenbench/dynahug` image
- `scripts/validate_oracle.py`, `organize_corpus.py`, `check_oracle_disjointness.py`

**Outputs**
- `real_benign_corpus/oracle-validation.json`
- `real_benign_corpus/oracle_positive/`, `oracle_negative/`
- `real_benign_corpus/oracle-split.json`

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), 'notebooks'))
from common import run, run_silent, sqlite, show, summary_line
run(["python3", "scripts/validate_oracle.py", "real_benign_corpus/all_pt",
     "--sample", "100", "--out", "real_benign_corpus/oracle-validation.json", "--backend", "docker", "--format", "pt", "--oracle-model-dir", "real_benign_corpus/oracle-calibrated/pt"])

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), 'notebooks'))
from common import run, run_silent, sqlite, show, summary_line
show("real_benign_corpus/oracle-validation.json", max_lines=40)

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), 'notebooks'))
from common import run, run_silent, sqlite, show, summary_line
run(["python3", "scripts/organize_corpus.py",
     "--corpus", "real_benign_corpus/all", "--report", "real_benign_corpus/oracle-validation.json", "--out", "real_benign_corpus"])

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), 'notebooks'))
from common import run, run_silent, sqlite, show, summary_line
run(["python3", "scripts/check_oracle_disjointness.py", "--resplit"])

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), 'notebooks'))
from common import run, run_silent, sqlite, show, summary_line
import json
s = json.load(open("real_benign_corpus/oracle-split.json"))
print("seed:", s["seed"], "| train:", len(s["train"]), "| eval:", len(s["eval"]))
assert set(s["train"]).isdisjoint(set(s["eval"])), "train/eval overlap!"
summary_line("disjoint 50/50 split", True)